# 01 · Data cleaning

**Input:** `data/raw/survey_responses_anonymised.csv` (46 responses, Google Form, November 2025)
**Output:** `data/clean/survey_clean.csv`

This notebook turns the raw questionnaire answers into a tidy numeric dataset. Every decision is made in code so
anyone can retrace it from the raw file, which the original Excel workflow did not allow.

**Order of work and why:** we look at every column before changing anything (discovery first), then rename, then
check value ranges, then convert text answers to numbers. Range checks come before conversion because a bad value
that slips through a conversion becomes an innocent-looking number that is much harder to spot later.

**Privacy note:** Timestamp, consent, gender and age were removed before this file was published. In a class of 46,
those columns together could identify individual students. All 46 respondents gave consent.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "survey_responses_anonymised.csv"
CLEAN = ROOT / "data" / "clean" / "survey_clean.csv"

raw = pd.read_csv(RAW)
raw.shape

(46, 29)

## 1. First look at every column

We do not choose variables yet. We look at what each column actually contains, because column names in survey
exports are often misleading (you will see one below).

Columns starting with `orig_` are the values the team originally calculated in Excel. They are kept only so the
analysis notebook can compare them with the rebuilt versions. They are **not** used as inputs.

In [2]:
for col in raw.columns:
    if col == "respondent_id":
        continue
    print(f"--- {col}")
    print(raw[col].value_counts(dropna=False).sort_index().to_dict(), "\n")

--- 3) Year of Study
{'1st Year': 40, '2nd year': 6} 

--- 4) Academic Performance ( According to you )
{'50% to 60%': 4, '60% to 70%': 5, '70% to 80%': 11, '80% to 90%': 16, 'Above 90%': 7, 'Below 50%': 3} 

--- 5) I enjoy working in situations involving competitions
{1: 1, 2: 3, 3: 15, 4: 17, 5: 10} 

--- 6) It is important for me to perform better than other people on a task.
{1: 1, 2: 6, 3: 15, 4: 11, 5: 13} 

--- 7) I try harder when I compete with other people.
{1: 2, 2: 4, 3: 10, 4: 17, 5: 13} 

--- 5) I enjoy working in groups and sharing ideas.
{1: 3, 3: 13, 4: 15, 5: 12, 7: 1, 8: 2} 

--- 9) I often share my notes and resources with classmates.
{1: 3, 3: 6, 4: 14, 5: 23} 

--- 10) I feel comfortable relying on others in group work.
{1: 10, 2: 10, 3: 12, 4: 10, 5: 4} 

--- 11) How often do your Assignments involve group work where the group is graded together?
{'Always': 2, 'Often': 18, 'Rarely': 5, 'Sometimes': 21} 

--- orig_Grouip_Assignment_Involvement_frequency
{2: 5, 3: 

**What the first look shows:**

1. The question "I enjoy working in groups and sharing ideas" is labelled **5)** in the header, but it is
   actually question 8. Questions 5 to 7 are about competition and 8 to 10 are about cooperation.
2. That same question contains the values **7 and 8**, which cannot exist on a 1 to 5 agreement scale. These are
   data-entry errors.
3. Questions 15 (relative performance) and 2 (age, already removed) allowed **more than one answer**, for example
   "Average, Above Average".
4. Question 16 has one blank answer.
5. The study-frequency options are Never, Once, 2-3 Times, Weekly, Daily and Always. The original Excel coding put
   Always (4) **below** Daily (5), which does not respect the order of the answers.

## 2. Rename columns

Short names make the code readable and stop the wrong-label problem above from spreading. The mapping lives in one
dictionary so a reviewer can check every rename in one place.

In [3]:
RENAME = {
    "3) Year of Study": "year_of_study",
    "4) Academic Performance ( According to you )": "marks_band_text",
    "5) I enjoy working in situations involving competitions": "q5_enjoy_competition",
    "6) It is important for me to perform better than other people on a task.": "q6_outperform_others",
    "7) I try harder when I compete with other people.": "q7_try_harder_competing",
    "5) I enjoy working in groups and sharing ideas.": "q8_enjoy_group_work",   # mislabelled as 5) in the export
    "9) I often share my notes and resources with classmates.": "q9_share_notes",
    "10) I feel comfortable relying on others in group work.": "q10_rely_on_others",
    "11) How often do your Assignments involve group work where the group is graded together?": "group_graded_text",
    "12) In the last month, how often did you study with classmates?": "study_with_peers_text",
    "13) Have you observed Free-Riding (classmates not contributing in group tasks)": "free_riding_text",
    "14) How motivated are you to study for graded assessments?": "q14_motivation",
    "15) How would you rate your academic performance relative to your classmates?": "relative_perf_text",
    "16) In your experience, what helps you learn better, cooperation with classmates or competition?": "learns_better_with",
}
df = raw.rename(columns=RENAME)

# Fail loudly if the export ever changes and a rename silently misses
missing = set(RENAME.values()) - set(df.columns)
assert not missing, missing
df.columns.tolist()

['respondent_id',
 'year_of_study',
 'marks_band_text',
 'q5_enjoy_competition',
 'q6_outperform_others',
 'q7_try_harder_competing',
 'q8_enjoy_group_work',
 'q9_share_notes',
 'q10_rely_on_others',
 'group_graded_text',
 'orig_Grouip_Assignment_Involvement_frequency',
 'study_with_peers_text',
 'orig_Study_Frequency_with_Peers',
 'free_riding_text',
 'orig_FREE_RIDING',
 'q14_motivation',
 'relative_perf_text',
 'learns_better_with',
 'orig_Cooperation_Index',
 'orig_Competition_Index',
 'orig_coop_index',
 'orig_comp_index',
 'orig_High_Cooperation_index',
 'orig_High_Competition_index',
 'orig_Belonging_Index',
 'orig_Reward',
 'orig_Sucker',
 'orig_Temptation',
 'orig_Punishment']

## 3. Range checks on the 1 to 5 agreement items

**Hypothesis being tested:** every Likert item only contains 1, 2, 3, 4 or 5.
**Decision (data-driven):** values outside 1 to 5 are set to missing rather than guessed. We cannot know whether a
7 was meant to be a 5 or a 1, and inventing a value would be worse than losing one item for that student.
The student's index is then averaged over the remaining items, so we do not lose the whole respondent.

In [4]:
LIKERT = ["q5_enjoy_competition", "q6_outperform_others", "q7_try_harder_competing",
          "q8_enjoy_group_work", "q9_share_notes", "q10_rely_on_others", "q14_motivation"]

out_of_range = df[LIKERT].apply(lambda s: (~s.between(1, 5)).sum())
print("Out-of-range values per item:")
print(out_of_range[out_of_range > 0])
print("\nAffected respondents:")
print(df.loc[~df["q8_enjoy_group_work"].between(1, 5), ["respondent_id", "q8_enjoy_group_work"]])

Out-of-range values per item:
q8_enjoy_group_work    3
dtype: int64

Affected respondents:
    respondent_id  q8_enjoy_group_work
42             43                    7
43             44                    8
45             46                    8


In [5]:
for col in LIKERT:
    df[col] = df[col].where(df[col].between(1, 5))   # anything outside 1-5 becomes NaN

vals = df[LIKERT].to_numpy().ravel()
assert pd.Series(vals).dropna().between(1, 5).all()   # works on pandas 2 and 3
df[LIKERT].describe().round(2)

,q5_enjoy_competition,q6_outperform_others,q7_try_harder_competing,q8_enjoy_group_work,q9_share_notes,q10_rely_on_others,q14_motivation
count,46.00,46.00,46.00,43.00,46.00,46.00,46.00
mean,3.70,3.63,3.76,3.77,4.17,2.74,3.89
std,0.96,1.10,1.10,1.09,1.10,1.27,1.16
min,1.00,1.00,1.00,1.00,1.00,1.00,1.00
25%,3.00,3.00,3.00,3.00,4.00,2.00,3.25
50%,4.00,4.00,4.00,4.00,4.50,3.00,4.00
75%,4.00,5.00,5.00,5.00,5.00,4.00,5.00
max,5.00,5.00,5.00,5.00,5.00,5.00,5.00


## 4. Convert text answers to numbers

Each mapping is written out as an explicit dictionary so the ordering is visible and can be challenged. After each
mapping we assert that nothing was left unmapped, because `.map()` silently returns NaN for any answer it does not
recognise, which is one of the most common silent bugs in survey cleaning.

**Multi-answer rule (domain decision):** where a student ticked two answers to a single-choice question, we set it
to missing. Averaging two ordinal categories would invent an answer the student never gave.

In [6]:
MARKS_BAND = {"Below 50%": 1, "50% to 60%": 2, "60% to 70%": 3,
              "70% to 80%": 4, "80% to 90%": 5, "Above 90%": 6}
RELATIVE_PERF = {"Below Average": 1, "little Below Average": 2, "Average": 3,
                 "Above Average": 4, "Well above average": 5}
GROUP_GRADED = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
STUDY_WITH_PEERS = {"Never": 0, "Once": 1, "2-3 Times": 2, "Weekly": 3, "Daily": 4, "Always": 5}
FREE_RIDING = {"No": 0, "Yes": 1}          # "Not Sure" is kept as its own category, not forced to 0.5
YEAR = {"1st Year": 1, "2nd year": 2}

def map_strict(series, mapping, allow_multi_to_nan=False):
    s = series.str.strip()
    if allow_multi_to_nan:
        s = s.where(~s.str.contains(",", na=False))   # multi-answers -> NaN
    mapped = s.map(mapping)
    unmapped = s[s.notna() & mapped.isna()].unique()
    assert len(unmapped) == 0, f"Unmapped answers: {unmapped}"
    return mapped

df["marks_band"] = map_strict(df["marks_band_text"], MARKS_BAND)
df["relative_perf"] = map_strict(df["relative_perf_text"], RELATIVE_PERF, allow_multi_to_nan=True)
df["group_graded_freq"] = map_strict(df["group_graded_text"], GROUP_GRADED)
df["study_with_peers"] = map_strict(df["study_with_peers_text"], STUDY_WITH_PEERS)
df["year_of_study"] = map_strict(df["year_of_study"], YEAR)

df["free_riding_text"] = df["free_riding_text"].str.strip()
df["free_riding"] = df["free_riding_text"].map(FREE_RIDING)        # NaN for "Not Sure", on purpose
df["free_riding_unsure"] = (df["free_riding_text"] == "Not Sure").astype(int)

print("Multi-answers set to missing in relative_perf:", df["relative_perf"].isna().sum())

Multi-answers set to missing in relative_perf: 2


**Why "Not Sure" on free-riding is not coded as 0.5:** the Excel version coded it as 0.5, which treats "I don't know"
as being halfway between "yes" and "no". That is an assumption, not a measurement. Keeping it as a separate flag
lets the analysis decide how to treat those 7 students and show whether the decision matters.

## 5. The open-preference question (Q16)

This is a multi-select question, so "Cooperation, Competition" is a genuine answer and is kept as **Both**.

In [7]:
PREF = {"Cooperation": "Cooperation", "Competition": "Competition", "Cooperation, Competition": "Both"}
df["learns_better_with"] = map_strict(df["learns_better_with"].fillna("").replace("", np.nan), PREF)
df["learns_better_with"].value_counts(dropna=False)

learns_better_with
Cooperation    29
Competition     9
Both            7
NaN             1
Name: count, dtype: int64

## 6. Missingness summary and save

We keep the cleaned answer columns and the `orig_` comparison columns, and drop the text versions that now have
numeric equivalents.

In [8]:
KEEP = (["respondent_id", "year_of_study", "marks_band", "relative_perf"] + LIKERT[:-1] +
        ["q14_motivation", "group_graded_freq", "study_with_peers", "free_riding", "free_riding_unsure",
         "free_riding_text", "learns_better_with"] +
        [c for c in df.columns if c.startswith("orig_")])
clean = df[KEEP].copy()

print("Missing values after cleaning:")
print(clean.drop(columns=[c for c in clean.columns if c.startswith("orig_")]).isna().sum().loc[lambda s: s > 0])

CLEAN.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(CLEAN, index=False)
print(f"\nSaved {clean.shape[0]} rows x {clean.shape[1]} columns to {CLEAN.relative_to(ROOT)}")

Missing values after cleaning:
relative_perf          2
q8_enjoy_group_work    3
free_riding            7
learns_better_with     1
dtype: int64

Saved 46 rows x 31 columns to data/clean/survey_clean.csv


## Summary of cleaning decisions

| Issue | Rows | Decision | Type of decision |
|---|---|---|---|
| Q8 labelled "5)" in export | all | Renamed to `q8_enjoy_group_work` | Data-driven |
| Q8 values of 7 and 8 | 3 | Set to missing | Data-driven |
| Two answers ticked on Q15 | 2 | Set to missing | Domain (single-choice question) |
| Free-riding "Not Sure" | 7 | Kept as separate flag, not 0.5 | Domain |
| Study frequency: Always vs Daily | 5 | Always (5) placed above Daily (4) | Domain (answer order) |
| Q16 blank | 1 | Left missing | Data-driven |
| Timestamp, consent, gender, age | all | Removed before publication | Privacy |